In [1]:
import os
import pandas as pd

# Create the folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Load raw, drop CustomerID, save clean version
df = pd.read_csv('../data/raw/customer_churn.csv')
df_clean = df.drop(columns=['CustomerID'])
df_clean.to_csv('../data/processed/churn_clean.csv', index=False)

# Verify it worked
print(f"✅ Saved successfully")
print(f"   Shape: {df_clean.shape}")
print(f"   Location: data/processed/churn_clean.csv")

✅ Saved successfully
   Shape: (440833, 11)
   Location: data/processed/churn_clean.csv


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/customer_churn.csv')

print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
#df.head()
df.columns.tolist()

Dataset shape: (440833, 12)

Column names: ['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']


['CustomerID',
 'Age',
 'Gender',
 'Tenure',
 'Usage Frequency',
 'Support Calls',
 'Payment Delay',
 'Subscription Type',
 'Contract Length',
 'Total Spend',
 'Last Interaction',
 'Churn']

In [5]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Basic Statistics ===")
df.describe()

=== Missing Values ===
CustomerID           1
Age                  1
Gender               1
Tenure               1
Usage Frequency      1
Support Calls        1
Payment Delay        1
Subscription Type    1
Contract Length      1
Total Spend          1
Last Interaction     1
Churn                1
dtype: int64

=== Data Types ===
CustomerID           float64
Age                  float64
Gender                object
Tenure               float64
Usage Frequency      float64
Support Calls        float64
Payment Delay        float64
Subscription Type     object
Contract Length       object
Total Spend          float64
Last Interaction     float64
Churn                float64
dtype: object

=== Basic Statistics ===


,CustomerID,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Churn
count,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000,440832.000000
mean,225398.667955,39.373153,31.256336,15.807494,3.604437,12.965722,631.616223,14.480868,0.567107
std,129531.918550,12.442369,17.255727,8.586242,3.070218,8.258063,240.803001,8.596208,0.495477
min,2.000000,18.000000,1.000000,1.000000,0.000000,0.000000,100.000000,1.000000,0.000000
25%,113621.750000,29.000000,16.000000,9.000000,1.000000,6.000000,480.000000,7.000000,0.000000
50%,226125.500000,39.000000,32.000000,16.000000,3.000000,12.000000,661.000000,14.000000,1.000000
75%,337739.250000,48.000000,46.000000,23.000000,6.000000,19.000000,830.000000,22.000000,1.000000
max,449999.000000,65.000000,60.000000,30.000000,10.000000,30.000000,1000.000000,30.000000,1.000000


In [6]:
churn_rate = df['Churn'].mean()
print(f"Overall churn rate:    {churn_rate:.1%}")
print(f"Churned customers:     {df['Churn'].sum():,}")
print(f"Retained customers:    {(df['Churn'] == 0).sum():,}")

Overall churn rate:    56.7%
Churned customers:     249,999.0
Retained customers:    190,833


In [7]:
features = [
    'Age', 'Tenure', 'Usage Frequency', 'Support Calls',
    'Payment Delay', 'Total Spend', 'Last Interaction'
]

# Mean of each feature, split by churned vs retained
comparison = df.groupby('Churn')[features].mean().round(2)
comparison.index = ['Retained (0)', 'Churned (1)']
print("=== Average values: Retained vs Churned ===")
print(comparison.to_string())

# The gap — this is what Agent 1 will use
print("\n=== Gap (Churned minus Retained) ===")
diff = comparison.loc['Churned (1)'] - comparison.loc['Retained (0)']
print(diff.sort_values(ascending=False).to_string())

=== Average values: Retained vs Churned ===
                Age  Tenure  Usage Frequency  Support Calls  Payment Delay  Total Spend  Last Interaction
Retained (0)  36.26   32.28            16.26           1.59          10.02       749.95             13.01
Churned (1)   41.75   30.47            15.46           5.14          15.22       541.29             15.60

=== Gap (Churned minus Retained) ===
Age                   5.49
Payment Delay         5.20
Support Calls         3.55
Last Interaction      2.59
Usage Frequency      -0.80
Tenure               -1.81
Total Spend        -208.66


In [8]:
print("=== Churn rate by Subscription Type ===")
print(df.groupby('Subscription Type')['Churn'].mean().round(3))

print("\n=== Churn rate by Contract Length ===")
print(df.groupby('Contract Length')['Churn'].mean().round(3))

print("\n=== Churn rate by Gender ===")
print(df.groupby('Gender')['Churn'].mean().round(3))

=== Churn rate by Subscription Type ===
Subscription Type
Basic       0.582
Premium     0.559
Standard    0.561
Name: Churn, dtype: float64

=== Churn rate by Contract Length ===
Contract Length
Annual       0.461
Monthly      1.000
Quarterly    0.460
Name: Churn, dtype: float64

=== Churn rate by Gender ===
Gender
Female    0.667
Male      0.491
Name: Churn, dtype: float64


In [9]:
signal_thresholds = {
    'high_support_calls':    4,  # Support Calls above this = warning
    'high_payment_delay':    13,  # Payment Delay (days) above this = warning
    'low_usage_frequency':   15,  # Usage Frequency below this = warning
    'low_tenure':            600,  # Tenure (months) below this = at risk
    'high_last_interaction': 14,  # Last Interaction (days) above this = disengaged
}

print("=== Proposed Agent 1 Signal Thresholds ===")
for signal, value in signal_thresholds.items():
    print(f"  {signal}: {value}")

=== Proposed Agent 1 Signal Thresholds ===
  high_support_calls: 4
  high_payment_delay: 13
  low_usage_frequency: 15
  low_tenure: 600
  high_last_interaction: 14


In [10]:
print("=== Threshold Validation ===")
print("(What % of churned customers does each threshold catch?)\n")

churned = df[df['Churn'] == 1]
retained = df[df['Churn'] == 0]

checks = [
    ('Support Calls >= 4',    churned['Support Calls'] >= 4),
    ('Payment Delay >= 13',   churned['Payment Delay'] >= 13),
    ('Last Interaction >= 15',churned['Last Interaction'] >= 15),
    ('Total Spend <= 600',    churned['Total Spend'] <= 600),
]

for label, condition in checks:
    detection_rate = condition.mean()
    
    # Also check false alarm rate (retained customers wrongly flagged)
    if '>=' in label:
        col, val = label.split(' >= ')
        false_alarm = (retained[col] >= int(val)).mean()
    else:
        col, val = label.split(' <= ')
        false_alarm = (retained[col] <= int(val)).mean()
    
    print(f"{label}")
    print(f"  Catches {detection_rate:.1%} of churned customers")
    print(f"  False alarms on {false_alarm:.1%} of retained customers")
    print()

=== Threshold Validation ===
(What % of churned customers does each threshold catch?)

Support Calls >= 4
  Catches 65.9% of churned customers
  False alarms on 9.1% of retained customers

Payment Delay >= 13
  Catches 58.9% of churned customers
  False alarms on 38.2% of retained customers

Last Interaction >= 15
  Catches 54.0% of churned customers
  False alarms on 37.9% of retained customers

Total Spend <= 600
  Catches 57.2% of churned customers
  False alarms on 20.0% of retained customers



In [11]:
# Add this to your notebook as a new cell
print("=== Refining Payment Delay threshold ===\n")

churned = df[df['Churn'] == 1]
retained = df[df['Churn'] == 0]

for threshold in [13, 15, 17, 19]:
    detection = (churned['Payment Delay'] >= threshold).mean()
    false_alarm = (retained['Payment Delay'] >= threshold).mean()
    print(f"Payment Delay >= {threshold}: catches {detection:.1%}, false alarms {false_alarm:.1%}")

=== Refining Payment Delay threshold ===

Payment Delay >= 13: catches 58.9%, false alarms 38.2%
Payment Delay >= 15: catches 52.6%, false alarms 28.6%
Payment Delay >= 17: catches 46.2%, false alarms 19.1%
Payment Delay >= 19: catches 39.9%, false alarms 9.6%


In [12]:
# ============================================================
# FINAL SIGNAL THRESHOLDS — Phase 1 output
# These feed directly into Agent 1 (Phase 3)
# ============================================================

SIGNAL_THRESHOLDS = {
    'high_support_calls':    4,    # 65.9% detection | 9.1%  false alarm ← STRONG
    'low_total_spend':       600,  # 57.2% detection | 20.0% false alarm ← GOOD
    'high_payment_delay':    17,   # 46.2% detection | 19.1% false alarm ← REFINED
    'high_last_interaction': 15,   # 54.0% detection | 37.9% false alarm ← COMBINE ONLY
    'low_usage_frequency':   14,   # weak — additive only
}

SIGNAL_WEIGHTS = {
    'high_support_calls':    0.35,
    'low_total_spend':       0.25,
    'high_payment_delay':    0.20,
    'high_last_interaction': 0.15,
    'low_usage_frequency':   0.05,
}

# Sanity check — weights must sum to 1.0
assert sum(SIGNAL_WEIGHTS.values()) == 1.0, "Weights must sum to 1.0"
print("Thresholds locked. Weights sum to:", sum(SIGNAL_WEIGHTS.values()))
print("\nReady for Agent 1 construction in Phase 3.")

Thresholds locked. Weights sum to: 1.0

Ready for Agent 1 construction in Phase 3.


In [ ]:
!pip install openai

In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

# Point the OpenAI client at OpenRouter instead
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# Your first LLM API call
response = client.chat.completions.create(
    model="anthropic/claude-3-haiku",  # cheap and fast for testing
    messages=[
        {
            "role": "user",
            "content": "Say hello and tell me you are working correctly. One sentence only."
        }
    ]
)

# Extract the response text
reply = response.choices[0].message.content
print("✅ API connected successfully")
print(f"Model says: {reply}")

✅ API connected successfully
Model says: Hello, I am working correctly.
